# F-S-0-RES: Reconstruction Error vs. Angular Separation

Single-frequency, noiseless angular-resolution check. This measures reconstruction/conditioning, not support detection.
### 0. Paramters and imports

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root(start=Path.cwd()):
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "src" / "cs_priors").exists():
            return path
    raise RuntimeError("Could not find repository root")


REPO_ROOT = find_repo_root()
sys.path[:0] = [
    str(REPO_ROOT / "src"),
    str(REPO_ROOT / "notebooks" / "benchmarks_v2" / "functions"),
]

from reconstruction_error import (
    plot_reconstruction_error_vs_separation,
    run_angle_separation_benchmark,
)


TAG = "F-S-0-RES"
FIGURE_DIR = REPO_ROOT / "results" / "benchmarks" / "v2" / TAG
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


def save_figure(fig, stem: str):
    fig.savefig(FIGURE_DIR / f"{stem}.pdf", dpi=300, bbox_inches="tight")


SEPARATIONS_DEG = np.array([
    1e-9, 3e-9,
    1e-8, 3e-8,
    1e-7, 3e-7,
    1e-6, 3e-6,
    1e-5, 3e-5,
    1e-4, 3e-4,
    6e-4, 9e-4,
    1e-3, 3e-3,
    1e-2, 3e-2,
    1e-1, 3e-1,
    1e0, 3e0,
    1e1, 3e1,
])
PHASE_SEEDS = np.arange(15)

SIM_KWARGS = dict(
    num_mics=2,
    num_sources=2,
    num_active=2,
    sampling_rate=20_000.0,
    duration=0.005,
    source_distance=1.5,
    mic_radius=0.3,
    mic_angle_start=0.0,
    source_angle_start=0.0,
    component_amplitude=1.0,
    magnitude_jitter=0.0,
    single_frequency_hz=1_000.0,
)

LASSO_ALPHA = 1e-5
LASSO_MAX_ITER = 10_000

### 1. Sanity check

In [ ]:
from cs_priors.plotting.plot_complex import plot_matrices
from cs_priors.plotting.plot_geometry import plot_sim_geometry
from reconstruction_error import make_simulation, solve_methods

sim = make_simulation(SEPARATIONS_DEG[-1], PHASE_SEEDS[0], **SIM_KWARGS)
solutions = solve_methods(sim)

from figures import save_pdf

fig, ax = plot_sim_geometry(sim, dpi=300, pad_factor=0.1, show=False)
save_pdf(fig, FIGURE_DIR / "simulation_geometry.pdf")


plot_matrices(
    [sim.X, *solutions.values()],
    ["X true", *solutions.keys()],
    show_values=True,
)


### 2. Generating results

In [ ]:
results_df = run_angle_separation_benchmark(
    SEPARATIONS_DEG,
    PHASE_SEEDS,
    sim_kwargs=SIM_KWARGS,
    lasso_alpha=LASSO_ALPHA,
    lasso_max_iter=LASSO_MAX_ITER,
)

results_df["method"] = pd.Categorical(
    results_df["method"], categories=["r-LASSO", "Sparse Prior", "X_pinv"], ordered=True
)
results_df.head()

### 3. Plotting results

In [ ]:
fig, ax, summary_df = plot_reconstruction_error_vs_separation(
    results_df,
    method_order=["r-LASSO", "Sparse Prior", "X_pinv"],
)
save_figure(fig, "relative_complex_error_vs_angle_separation")
plt.show()

summary_df.head()